# 통합 비교 결과 시각화

`compare_methods.py`의 출력 CSV를 읽는 기본 Matplotlib 그림
색상과 글꼴은 지정하지 않으며 학습 패키지를 불러오지 않음
오차막대는 시드별 값의 평균에 대한 t 분포 95% 신뢰구간이며 우월성 검정을 의미하지 않음
시험 평가를 명시한 결과는 시험 F1을 사용하고 그 외에는 검증 F1을 사용

In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t

working = Path.cwd().resolve()
candidates = [working, *working.parents]
candidates += [parent / "Federated_Learning" for parent in [working, *working.parents]]
project = next((parent for parent in candidates
                if (parent / "pyproject.toml").is_file()
                and (parent / "2_analysis_speed" / "2-4_graph_deduplication").is_dir()), working)
result_csv = project / "results" / "comparison" / "aggregate_metrics.csv"
with result_csv.open(encoding="utf-8-sig", newline="") as handle:
    rows = list(csv.DictReader(handle))
if not rows:
    raise ValueError("The comparison CSV has no completed runs")
methods = ["legacy_dynamic", "compact_dynamic", "transformer_static", "transformer_dynamic"]
labels = ["Legacy zlib/XOR", "Compact zlib/XOR", "Transformer/Static", "Transformer/Dynamic"]
datasets = sorted({row["dataset"] for row in rows})
quality_key = "test_macro_f1" if all(row.get("test_macro_f1") not in (None, "", "None") for row in rows) else "selected_val_macro_f1"
quality_label = "Test macro F1 (%)" if quality_key == "test_macro_f1" else "Validation macro F1 (%)"
def estimate(values):
    values = np.asarray(values, dtype=float)
    if not values.size or not np.isfinite(values).all():
        raise ValueError("Expected finite measurements for every comparison arm")
    interval = t.ppf(0.975, len(values)-1) * values.std(ddof=1) / np.sqrt(len(values)) if len(values)>1 else 0.0
    return values.mean(), interval

In [ ]:
fig, axes = plt.subplots(2, len(datasets), figsize=(6 * len(datasets), 7), squeeze=False)
for column, dataset in enumerate(datasets):
    traffic, traffic_ci, quality, quality_ci = [], [], [], []
    for method in methods:
        selected = [row for row in rows if row["dataset"] == dataset and row["method"] == method]
        mean, interval = estimate([float(row["sum_total_bytes"]) / 1e6 for row in selected])
        traffic.append(mean)
        traffic_ci.append(interval)
        mean, interval = estimate([100 * float(row[quality_key]) for row in selected])
        quality.append(mean)
        quality_ci.append(interval)
    axes[0, column].barh(labels, traffic, xerr=traffic_ci)
    axes[0, column].invert_yaxis()
    axes[0, column].set_xlabel("Cumulative application traffic (MB)")
    axes[0, column].set_title(dataset)
    axes[1, column].errorbar(quality, np.arange(len(labels)), xerr=quality_ci, fmt="o", capsize=3)
    axes[1, column].set_yticks(np.arange(len(labels)), labels=labels)
    axes[1, column].invert_yaxis()
    axes[1, column].set_xlabel(quality_label)
fig.tight_layout()
plt.show()

필요한 경우 아래 셀로 그림 파일 저장
노트북 실행 출력과 저장 그림은 소스 배포 ZIP에서 제외